# NHS Outpatient Referral Data Engineering Project

### End-to-End ETL Pipeline Using Official NHS Data

## Project Overview

This project demonstrates the design and implementation of an end-to-end data engineering pipeline using official NHS outpatient referral data.

The objective is to transform raw monthly referral data into a trusted, analysis-ready dataset that supports operational reporting and decision-making.

The completed solution will include:

- Data exploration and profiling
- Data quality assessment
- Python ETL pipeline
- PostgreSQL database
- SQL reporting layer
- Power BI dashboard
- Project documentation
## Phase 1: Data Exploration

### Project Objective

The objective of this notebook is to understand the structure, quality, and business context of the NHS Outpatient Referral dataset before designing the ETL pipeline.

Rather than moving directly into data transformation, this exploration focuses on understanding what the data represents, identifying potential data quality issues, and documenting findings that will inform the database design and ETL process.

---

## Business Context

NHS organisations publish outpatient referral data to support operational reporting and service planning.

The Business Intelligence team requires a repeatable ETL process that transforms raw referral data into a trusted, analysis-ready dataset for reporting and decision-making.

This notebook represents the discovery phase of that solution.

---

## Objectives

During this exploration, the following questions will be investigated:

- What does one record represent?
- What is the grain of the dataset?
- Which columns uniquely identify a business record?
- Which columns are descriptive attributes?
- Which columns contain business measures?
- Are there missing values or duplicate records?
- What data quality rules should be applied during the ETL process?

---

## Dataset

**Source:** NHS England

**Reporting Period:** April 2023 – March 2024

**File:** `outpatient_referrals_2023_24.csv`

---

## Expected Outputs

By the end of this notebook we should have:

- A clear understanding of the dataset structure
- The identified business key
- A documented data dictionary
- Initial data quality findings
- Recommendations for the ETL pipeline
- Inputs required for the PostgreSQL data model


In [2]:
from pathlib import Path

import pandas as pd

In [4]:
DATA_PATH = Path("../data/raw/outpatient_referrals_2023_24.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")

Dataset loaded successfully


In [7]:
df.head()

,Periodname,Provider Parent org code,Provider Parent name,Provider Org code,Provider Org name,Commissioner Parent Org Code,Commissioner Parent Org Name,Commissioner Org Code,Commissioner Org Name,Op Gprefsmade M,Op Otherrefsmade M,Op Gprefsmade Ga M,Op Otherrefsmade Ga M
0,MRR-April-2023,NaN,NaN,7A1,BETSI CADWALADR UNIVERSITY LHB,QYG,NHS CHESHIRE AND MERSEYSIDE INTEGRATED CARE BOARD,02E,NHS WARRINGTON (SUB ICB LOCATION),0,0,0,0
1,MRR-April-2023,NaN,NaN,7A1,BETSI CADWALADR UNIVERSITY LHB,QYG,NHS CHESHIRE AND MERSEYSIDE INTEGRATED CARE BOARD,12F,NHS WIRRAL (SUB ICB LOCATION),0,0,0,0
2,MRR-April-2023,NaN,NaN,7A1,BETSI CADWALADR UNIVERSITY LHB,QYG,NHS CHESHIRE AND MERSEYSIDE INTEGRATED CARE BOARD,27D,NHS CHESHIRE (SUB ICB LOCATION),10,0,10,0
3,MRR-April-2023,NaN,NaN,Y7R5Y,SPAMEDICA SWANSEA,QU9,"NHS BUCKINGHAMSHIRE, OXFORDSHIRE AND BERKSHIRE...",10Q,NHS OXFORDSHIRE (SUB ICB LOCATION),0,0,0,0
4,MRR-April-2023,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,00Q,NHS BLACKBURN WITH DARWEN (SUB ICB LOCATION),10,0,10,0


In [9]:
df.groupby(
    [
        "Periodname",
        "Provider Org code",
        "Commissioner Org Code"
    ]
).size().sort_values(ascending=False).head(20)

Periodname      Provider Org code  Commissioner Org Code
MRR-April-2023  7A1                02E                      1
                                   12F                      1
                                   27D                      1
                A0C5S              11M                      1
                                   15C                      1
                                   18C                      1
                A4M8P              00Q                      1
                                   00R                      1
                                   00T                      1
                                   00X                      1
                                   01A                      1
                                   01E                      1
                                   01V                      1
                                   02G                      1
                                   02H                      1
             

### Finding 1: Dataset grain

Investigation using the combination of `Periodname`, `Provider Org code`, and `Commissioner Org Code` showed that each combination occurs only once.

This indicates that one record represents the outpatient referral activity between a single commissioner and provider for a specific reporting period.

This combination is a candidate business key and will be validated further during the ETL design.

## Referral Measure Analysis

The dataset contains four numeric referral measures. This section examines their data types, completeness, ranges and relationships before transformation rules are defined.

The source publication describes the Monthly Referral Return as covering GP and other referrals for first consultant-led outpatient appointments.

The precise distinction between the `M` and `Ga M` measures will be confirmed against the accompanying NHS metadata before final column names are assigned.

In [11]:
referral_columns = [
    "Op Gprefsmade M",
    "Op Otherrefsmade M",
    "Op Gprefsmade Ga M",
    "Op Otherrefsmade Ga M",
]

df[referral_columns].dtypes

Op Gprefsmade M          int64
Op Otherrefsmade M       int64
Op Gprefsmade Ga M       int64
Op Otherrefsmade Ga M    int64
dtype: object

In [12]:
df[referral_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Op Gprefsmade M,99596.0,130.290795,847.978821,0.0,0.0,1.0,4.0,21332.0
Op Otherrefsmade M,99596.0,106.511386,691.733437,0.0,1.0,1.0,6.0,29601.0
Op Gprefsmade Ga M,99596.0,122.733092,813.674590,0.0,0.0,1.0,3.0,21326.0
Op Otherrefsmade Ga M,99596.0,94.982088,611.790375,0.0,0.0,1.0,5.0,26471.0


### Finding 2: Distribution of Referral Measures

The referral measures exhibit a highly skewed distribution.

Most provider–commissioner combinations record relatively low referral volumes, while a small number of combinations record substantially higher values.

This pattern is consistent with the expectation that larger NHS providers or specialist organisations receive referrals from much larger populations.

In [13]:
df[referral_columns].isna().sum()

Op Gprefsmade M          0
Op Otherrefsmade M       0
Op Gprefsmade Ga M       0
Op Otherrefsmade Ga M    0
dtype: int64

In [14]:
negative_value_counts = (
    df[referral_columns]
    .lt(0)
    .sum()
    .rename("negative_count")
)

negative_value_counts

Op Gprefsmade M          0
Op Otherrefsmade M       0
Op Gprefsmade Ga M       0
Op Otherrefsmade Ga M    0
Name: negative_count, dtype: int64

### Finding 3: Referral Count Validation

No negative referral values were identified in the dataset.

Referral measures represent counts of outpatient referrals and therefore must be greater than or equal to zero.

This rule will become a mandatory validation step in the ETL pipeline.

In [19]:
measure_comparison = pd.DataFrame({
    "gp_values_equal": (
        df["Op Gprefsmade M"]
        == df["Op Gprefsmade Ga M"]
    ),
    "other_values_equal": (
        df["Op Otherrefsmade M"]
        == df["Op Otherrefsmade Ga M"]
    ),
})

measure_comparison.mean()

gp_values_equal       0.910047
other_values_equal    0.820103
dtype: float64

### Finding 4: Comparison of Referral Measures

The paired GP referral measures are identical in approximately **91%** of records, while the paired Other Referral measures are identical in approximately **82%** of records.

Because the measures are not identical across all records, they should be treated as distinct fields until their business definitions are confirmed using the NHS documentation.

No transformation or consolidation of these measures will be performed during the initial ETL design.

gp_mismatches = df.loc[
    df["Op Gprefsmade M"] != df["Op Gprefsmade Ga M"],
    [
        "Periodname",
        "Provider Org code",
        "Commissioner Org Code",
        "Op Gprefsmade M",
        "Op Gprefsmade Ga M",
    ],
]

gp_mismatches.head(10)

In [20]:
gp_mismatches = df.loc[
    df["Op Gprefsmade M"] != df["Op Gprefsmade Ga M"],
    [
        "Periodname",
        "Provider Org code",
        "Commissioner Org Code",
        "Op Gprefsmade M",
        "Op Gprefsmade Ga M",
    ],
]

gp_mismatches.head(10)

,Periodname,Provider Org code,Commissioner Org Code,Op Gprefsmade M,Op Gprefsmade Ga M
156,MRR-April-2023,RXL,00R,2942,2903
162,MRR-April-2023,RXL,02M,1773,1769
197,MRR-April-2023,RXN,00X,2119,2114
199,MRR-April-2023,RXN,01E,2429,2424
254,MRR-April-2023,RXR,00Q,1571,1408
255,MRR-April-2023,RXR,00R,16,15
256,MRR-April-2023,RXR,00X,23,21
257,MRR-April-2023,RXR,01A,3548,3165
258,MRR-April-2023,RXR,01E,40,37
264,MRR-April-2023,RXR,15M,1,0


### Finding 5: Comparison of GP Referral Measures

The GP referral measures are not identical across all records.

Where differences occur, the values in `Op Gprefsmade Ga M` are consistently less than or equal to the corresponding values in `Op Gprefsmade M`.

The size of the differences varies considerably between records, suggesting that the two columns represent distinct business measures rather than duplicate data.

The business definitions of these measures should be confirmed using the NHS data dictionary before any transformation or aggregation is applied.

# Initial Findings

The exploratory analysis of the NHS Outpatient Referral dataset produced the following findings:

1. The dataset contains 99,596 records across 13 source fields.

2. Each record represents aggregated outpatient referral activity for one reporting period, one provider organisation and one commissioner organisation.

3. The candidate composite business key consists of:
   - Periodname
   - Provider Org code
   - Commissioner Org Code

4. All referral measure columns are complete and contain no missing values.

5. No negative referral counts were identified.

6. Referral measures exhibit a highly skewed distribution, with most provider–commissioner combinations recording relatively low referral volumes and a small number recording substantially higher volumes.

7. The paired referral measures are closely related but are not identical. Their business definitions should be confirmed using NHS metadata before any transformation or consolidation is applied.

8. Organisation codes should be used as identifiers during database modelling, while organisation names should be treated as descriptive attributes.

9. These findings will inform the design of the ETL pipeline and PostgreSQL data model.